# 面试题：重试和指数退避怎样避免放大故障？

回答要点：只重试暂时性、幂等或可安全去重的失败。采用指数退避、确定性抖动、最大尝试、总 deadline 和 retry budget；参数错误、权限拒绝和业务冲突不重试。每次尝试记录原因、等待和幂等键。超时是未知结果，写操作应先回读权威状态。

## 真实案例

库存服务处理六个请求，响应序列中包含 503、429、400、超时和成功。

## 基线

基线收到任何非 200 都在下一秒立即重发。

## 结果解读

手写策略输出每个请求的尝试时间、是否重试及停止原因。

## 失败案例

把 400 参数错误当成暂态故障，会白白放大下游请求。

In [1]:
requests = [{'id':'T01','responses':[503,200]}, {'id':'T02','responses':[429,429,200]}, {'id':'T03','responses':[400]}, {'id':'T04','responses':['timeout',200]}, {'id':'T05','responses':[503,503,503]}, {'id':'T06','responses':[200]}]  # 构造六条具有预设服务响应序列的库存查询事件。
print('响应序列输入:', requests)  # 输出每条请求可能经历的失败和恢复过程。
print('教学说明：时间单位是离散秒，抖动为确定性函数以保证 Notebook 可复放。')  # 说明受控时间模型的边界。

响应序列输入: [{'id': 'T01', 'responses': [503, 200]}, {'id': 'T02', 'responses': [429, 429, 200]}, {'id': 'T03', 'responses': [400]}, {'id': 'T04', 'responses': ['timeout', 200]}, {'id': 'T05', 'responses': [503, 503, 503]}, {'id': 'T06', 'responses': [200]}]
教学说明：时间单位是离散秒，抖动为确定性函数以保证 Notebook 可复放。


In [2]:
def immediate_retry(row):  # 定义任何错误每秒重试一次的放大故障基线。
    return list(range(len(row['responses'])))  # 返回所有尝试发生在连续秒上的时间表。
baseline = [(row['id'], immediate_retry(row)) for row in requests]  # 对六条请求生成立即重试时间表。
print('立即重试基线:', baseline)  # 输出同步重试会形成的请求尖峰。

立即重试基线: [('T01', [0, 1]), ('T02', [0, 1, 2]), ('T03', [0]), ('T04', [0, 1]), ('T05', [0, 1, 2]), ('T06', [0])]


In [3]:
def retryable(response):  # 定义哪些错误在教学策略中可以安全重试。
    return response in {429,503,'timeout'}  # 仅把限流、暂态不可用和超时标记为可重试。
def schedule(row, max_attempts=3, deadline=10):  # 定义带退避、抖动和总 deadline 的调度器。
    times = [0]  # 记录第一次调用在零时刻发出。
    for attempt, response in enumerate(row['responses'][:-1], start=1):  # 逐次观察除了末尾之外的响应。
        if not retryable(response) or attempt >= max_attempts:  # 遇到永久错误或达到预算时停止。
            return times, '停止:' + str(response)  # 返回已有计划与停止原因。
        wait = (2 ** (attempt - 1)) + (attempt % 2)  # 计算指数退避并加入可复放的确定性抖动。
        if times[-1] + wait > deadline:  # 防止重试穿透用户总 deadline。
            return times, '停止:deadline'  # 返回由于 deadline 终止的结果。
        times.append(times[-1] + wait)  # 记录下一次实际可发送时间。
    return times, '完成或待权威回读'  # 返回成功序列或超时需回读的状态。

In [4]:
results = [(row['id'],) + schedule(row) for row in requests]  # 为六条请求计算受预算限制的重试计划。
print('id | 尝试时间 | 终止原因')  # 输出重试账本标题。
for item in results:  # 遍历每条请求的计划与停止理由。
    print(item[0], item[1], item[2])  # 输出可审计的退避过程。
print('总额外重试数:', sum(len(item[1]) - 1 for item in results))  # 汇总重试带来的额外负载。

id | 尝试时间 | 终止原因
T01 [0, 2] 完成或待权威回读
T02 [0, 2, 4] 完成或待权威回读
T03 [0] 完成或待权威回读
T04 [0, 2] 完成或待权威回读
T05 [0, 2, 4] 完成或待权威回读
T06 [0] 完成或待权威回读
总额外重试数: 6


In [5]:
wrong_times = immediate_retry(requests[2])  # 对 400 参数错误应用错误的立即重试基线。
fixed_times, fixed_reason = schedule(requests[2])  # 对同一错误应用按语义分类的策略。
print('失败案例 T03：错误基线时间=', wrong_times, '，修正时间=', fixed_times, '，原因=', fixed_reason)  # 展示永久错误应立即返回结构化校验信息。
print('生产差距：真实策略还需服务端 Retry-After、全局 retry budget、并发限制、幂等键和超时后的状态回读。')  # 说明调度器未覆盖的生产控制项。

失败案例 T03：错误基线时间= [0] ，修正时间= [0] ，原因= 完成或待权威回读
生产差距：真实策略还需服务端 Retry-After、全局 retry budget、并发限制、幂等键和超时后的状态回读。


In [6]:
assert schedule(requests[2])[0] == [0]  # 验证 400 参数错误不会被重试。
assert schedule(requests[1])[0] == [0, 2, 4]  # 验证 429 使用指数退避和抖动后的时间表。
assert len(results[4][1]) == 3  # 验证持续 503 会受到最大尝试次数限制。